TODO :
- Decide what one row represents and write it down (for example: a repository, a film, a weather station, or an article). As Exercises 6 and 7 showed, this choice determines what your clusters mean.

    - A row is a game or stream. Our features are viewer counts, tag one-hot, hour-of-day profile

- Build the feature table X. The table below suggests features for each project source. Use one-hot encoding for categories, take the log of skewed counts, and use TF-IDF for text columns.

- Standardise the features ((feat - feat.mean()) / feat.std()), then cluster for two or three values of 𝑘. If you compute silhouette scores on more than a few thousand rows, pass sample_size=10_000 to silhouette_score so that the 𝑂(𝑛2) pairwise distance calculation does not hang.

- Print the cluster profile table and give every cluster a short descriptive name. Put the cluster names and interpretation in your report, and the full profile table in the appendix.

- Inspect any small or hard-to-name cluster. As with the 14 broken-meter trips in Exercise 6, an unusual cluster often reveals a data-quality issue missed in the previous weeks.

In [34]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.width", 100)
sns.set_theme(style="whitegrid")

CLEAN_DATA_DIR = Path.cwd().parent / "data/clean/"

clean_games_path = CLEAN_DATA_DIR / "games_cleaned.parquet"
clean_clips_path = CLEAN_DATA_DIR / "clips_cleaned.parquet"
clean_channels_path = CLEAN_DATA_DIR / "channels_cleaned.parquet"

cgdf = pd.read_parquet(clean_games_path)
cdf = pd.read_parquet(clean_clips_path)
chdf = pd.read_parquet(clean_channels_path)

### Feature table

In [35]:
# One-hot encoding and bag of words
# one-hot encode the content_classification_labels column
import re

# extract non-empty labels
label_set = {
    label
    for text in chdf['content_classification_labels'].dropna()
    for label in re.findall(r"'([^']+)'", str(text))
    if label.strip() not in ("[]", "")
}

for label in label_set:
    chdf[f"label_{label}"] = (
        chdf["content_classification_labels"]
        .astype("string")
        .str.contains(f"'{label}'", regex=False, na=False)
        .astype("int8")
    )

# label_none = 1 if all category columns sum to 0
label_cols = [f"label_{label}" for label in label_set]
chdf["label_none"] = (chdf[label_cols].sum(axis=1) == 0).astype("int8")

chdf = chdf.drop(columns=["content_classification_labels"])

print(chdf[:5])

   broadcaster_id broadcaster_language  game_id          game_name  \
0       234973404                   en   509658      Just Chatting   
1       683970274                   en   509658      Just Chatting   
2       552120296                   en    18122  World of Warcraft   
3       423486275                   ru   509658      Just Chatting   
4           12826                   en   509658      Just Chatting   

                                               title  \
0  IRL Shanghai 🇨🇳 I'm back with @pokimane 🥰 Dres...   
1  SOUL FOOD COOKING - FRIED CHICKEN, MAC N CHEES...   
2  [DROPS ON] UNBANNED WOW FOREVER BETA FINAL DAY...   
3  ПОДГОТОВКА К ГРЯЗНОЙ ФОТОСЕССИИ С ДЕВОЧКАМИ)) ...   
4  Twitch Public Access (October 2026) | Celebrat...   

                                                tags  is_branded_content  label_MatureGame  \
0  [['Travel' 'food' 'lifestyle' 'Chinese中文' 'IRL...               False                 0   
1  [['american' 'girl' 'asian' 'canadian' 'Englis...  

Previously, we saw that we have lots of unique tags :

In [ ]:
# calculate number of unique tags
all_tags = chdf['tags'].explode()
tag_counts = all_tags.value_counts()
print("Unique tags found:", tag_counts.shape[0])

Keep in mind that most represented tags are language tags. The probelem is that if we add 50 tags, the clustering will be overwhelmingly driven by the tags and not the other columns.